<a href="https://colab.research.google.com/github/DannyBank/ai-disease-predictor/blob/main/DiseasePredictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q xgboost scikit-learn pandas numpy gradio kaggle

import os
# If you have kaggle.json, uncomment below to configure:
# os.makedirs('/root/.kaggle', exist_ok=True)
# !cp kaggle.json /root/.kaggle/
# !chmod 600 /root/.kaggle/kaggle.json

In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

# 1. DATA ACQUISITION & PREPROCESSING (Simulated robust pipeline)
print("Downloading and preparing dataset...")

# Downloading a comprehensive health indicators dataset from Kaggle
!kaggle datasets download -d alexteboul/diabetes-health-indicators-dataset -p .
!unzip -o -q diabetes-health-indicators-dataset.zip -d .

# Load dataset (Using Diabetes Health Indicators as our primary clinical baseline engine)
df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')

# For demonstration of a multi-disease system, we simulate targets if not present in a single file,
# or map available columns to our clinical prediction targets:
# Features used: HighBP, HighChol, BMI, Smoker, Stroke, HeartDiseaseorAttack, PhysActivity, GenHlth, MentHlth, PhysHlth, DiffWalk, Sex, Age, Education, Income
X = df.drop(columns=['Diabetes_binary'])
y_diabetes = df['Diabetes_binary']

# Simulating correlated risk targets for demonstration purposes based on clinical heuristics + noise
# (In production, replace these with distinct disease dataframes/tables)
np.random.seed(42)
y_hypertension = df['HighBP']
y_heart_failure = df['HeartDiseaseorAttack']
y_stroke = df['Stroke']
y_kidney_failure = ((df['GenHlth'] >= 4) & (df['HighBP'] == 1)).astype(int)

# 2. MODEL TRAINING (XGBoost Classifiers)
print("Training Clinical Decision Support Models...")

specific_features = ['HighBP', 'HighChol', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack',
                   'PhysActivity', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk',
                   'Sex', 'Age', 'Education', 'Income']

# Filter X to only include the specific features before scaling
X_filtered = df[specific_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)

X_train, X_test, yd_train, yd_test = train_test_split(X_scaled, y_diabetes, test_size=0.2, random_state=42)

# Train individual specialized models for each clinical condition
models = {
    "Diabetes": xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42),
    "Hypertension": xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42),
    "Heart Failure": xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42),
    "Stroke": xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42),
    "Kidney Failure": xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
}

# Fit models (using y_diabetes split for demo, or respective targets)
models["Diabetes"].fit(X_train, yd_train)
models["Hypertension"].fit(X_train, train_test_split(y_hypertension, test_size=0.2, random_state=42)[0])
models["Heart Failure"].fit(X_train, train_test_split(y_heart_failure, test_size=0.2, random_state=42)[0])
models["Stroke"].fit(X_train, train_test_split(y_stroke, test_size=0.2, random_state=42)[0])
models["Kidney Failure"].fit(X_train, train_test_split(y_kidney_failure, test_size=0.2, random_state=42)[0])

print("Models trained successfully!")

# 3. CLINICAL DECISION SUPPORT & PREDICTION ENGINE
def predict_clinical_risks(high_bp, high_chol, bmi, smoker, stroke_hist, heart_hist,
                           phys_activity, gen_hlth, ment_hlth, phys_hlth, diff_walk,
                           sex, age, education, income, real_time_hr, real_time_spo2):

    # Construct input vector matching training features
    input_data = np.array([[high_bp, high_chol, bmi, smoker, stroke_hist, heart_hist,
                            phys_activity, gen_hlth, ment_hlth, phys_hlth, diff_walk,
                            sex, age, education, income]])

    input_scaled = scaler.transform(input_data)

    # Get predictive probabilities
    results = {}
    for disease, model in models.items():
        prob = model.predict_proba(input_scaled)[0][1] * 100
        results[disease] = round(prob, 2)

    # Real-time physiological alert logic
    alerts = []
    if real_time_hr > 100 or real_time_hr < 60:
        alerts.append(f"⚠️ **Tachycardia/Bradycardia Warning**: Heart Rate is {real_time_hr} bpm.")
    if real_time_spo2 < 95:
        alerts.append(f"⚠️ **Hypoxia Alert**: SpO2 level is critically low at {real_time_spo2}%")

    if not alerts:
        alerts.append("✅ Real-time physiological vitals are within normal stable limits.")

    # Clinical Decision Support Recommendations
    recommendations = []
    if results["Diabetes"] > 50:
        recommendations.append("• **Diabetes Protocol**: Recommend HbA1c lab verification and dietary consultation.")
    if results["Hypertension"] > 50 or high_bp == 1:
        recommendations.append("• **Hypertension Protocol**: Initiate 24-hour ambulatory blood pressure monitoring.")
    if results["Heart Failure"] > 40 or results["Stroke"] > 40:
        recommendations.append("• **Cardiovascular Protocol**: Urgent cardiology referral and ECG/Echo evaluation advised.")
    if not recommendations:
        recommendations.append("• Routine annual screening recommended. No acute intervention required.")

    # Format output summary
    output_report = f"""
    ### 📊 Disease Probability Estimates
    * **Diabetes Risk:** {round(float(results['Diabetes']), 2)}%
    * **Hypertension Risk:** {round(float(results['Hypertension']), 2)}%
    * **Heart Failure Risk:** {round(float(results['Heart Failure']), 2)} %
    * **Stroke Risk:** {round(float(results['Stroke']), 2)}%
    * **Kidney Failure Risk:** {round(float(results['Kidney Failure']), 2)}%

    ### 🩺 Real-Time Physiological Status
    {'\n'.join(alerts)}

    ### 💡 Clinical Decision Support & Recommendations
    {'\n'.join(recommendations)}
    """
    return output_report

# 4. INTERACTIVE WEB USER INTERFACE (Gradio)
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏥 AI-Powered Clinical Disease Prediction & Decision Support System")
    gr.Markdown("Combine patient historical data and real-time physiological vitals for early screening and clinical guidance.")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🧬 Patient Medical History & Demographics")
            high_bp = gr.Dropdown([0, 1], label="High Blood Pressure (0 = No, 1 = Yes)", value=0)
            high_chol = gr.Dropdown([0, 1], label="High Cholesterol (0 = No, 1 = Yes)", value=0)
            bmi = gr.Slider(10, 50, value=25, label="Body Mass Index (BMI)")
            smoker = gr.Dropdown([0, 1], label="Smoker (At least 5 packs in lifetime) (0 = No, 1 = Yes)", value=0)
            stroke_hist = gr.Dropdown([0, 1], label="History of Stroke (0 = No, 1 = Yes)", value=0)
            heart_hist = gr.Dropdown([0, 1], label="History of Heart Disease/Attack (0 = No, 1 = Yes)", value=0)
            phys_activity = gr.Dropdown([0, 1], label="Physical Activity in Past 30 Days (0 = No, 1 = Yes)", value=1)
            gen_hlth = gr.Slider(1, 5, step=1, value=3, label="General Health Scale (1 = Excellent, 5 = Poor)")
            ment_hlth = gr.Slider(0, 30, value=0, label="Days of Poor Mental Health (Past 30 days)")
            phys_hlth = gr.Slider(0, 30, value=0, label="Days of Poor Physical Health (Past 30 days)")
            diff_walk = gr.Dropdown([0, 1], label="Difficulty Walking / Climbing Stairs (0 = No, 1 = Yes)", value=0)
            sex = gr.Dropdown([0, 1], label="Sex (0 = Female, 1 = Male)", value=0)
            age = gr.Slider(1, 13, step=1, value=6, label="Age Category Scale (1: 18-24 ... 13: 85+) Residue")
            education = gr.Slider(1, 6, step=1, value=4, label="Education Level (1 to 6)")
            income = gr.Slider(1, 8, step=1, value=5, label="Income Scale (1 to 8)")

        with gr.Column():
            gr.Markdown("### 📡 Real-Time Physiological Stream Vitals")
            real_time_hr = gr.Slider(40, 180, value=75, label="Real-Time Heart Rate (bpm)")
            real_time_spo2 = gr.Slider(70, 100, value=98, label="Real-Time Blood Oxygen (SpO2 %)")

            submit_btn = gr.Button("Run AI Clinical Assessment", variant="primary", scale=2)

            gr.Markdown("### 📋 Clinical Output & Decision Support")
            output_display = gr.Markdown()

    submit_btn.click(
        fn=predict_clinical_risks,
        inputs=[high_bp, high_chol, bmi, smoker, stroke_hist, heart_hist,
                phys_activity, gen_hlth, ment_hlth, phys_hlth, diff_walk,
                sex, age, education, income, real_time_hr, real_time_spo2],
        outputs=output_display
    )

# Launch app directly inside Colab with a public shareable link
demo.launch(inline=True, share=True)

Dataset URL: https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset
License(s): CC0-1.0
100% 6.03M/6.03M [00:00<00:00, 87.0MB/s]

Training Clinical Decision Support Models...
Models trained successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://93202f19c2a81c2038.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
